
## Folder 05 — File 04: Business Rule Validation:


We will validate:

- Order total against order-item totals.
- Payment amounts against order totals.
- Multiple payment records for an order.
- Shipment records against orders.
- Delivered shipment date consistency.
- Inventory quantities and snapshots.
- Product prices against product costs.
- Orders with missing order items.


**Important:**
 These are business rules, not universal truths. For example, an order can legitimately have multiple partial payments. We should not automatically flag every multiple-payment order as invalid.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

TABLES = {
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "products": f"{CATALOG}.{SCHEMA}.silver_products",
    "orders": f"{CATALOG}.{SCHEMA}.silver_orders",
    "order_items": f"{CATALOG}.{SCHEMA}.silver_order_items",
    "payments": f"{CATALOG}.{SCHEMA}.silver_payments",
    "shipments": f"{CATALOG}.{SCHEMA}.silver_shipments"
}

# Inventory table resolution

# Prefer the full source dataset name.

INVENTORY_TABLE_CANDIDATES = [
    f"{CATALOG}.{SCHEMA}.silver_inventory_snapshots",
    f"{CATALOG}.{SCHEMA}.silver_inventory"
]

REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.business_rule_validation_report"
)

TOLERANCE = 0.02

print("Project configuration completed.")


## 3. Validate and load Silver datasets

for name, table_name in TABLES.items():

    if not spark.catalog.tableExists(table_name):
        raise Exception(
            f"Required Silver table not found: {table_name}"
        )

# Resolve inventory table.

INVENTORY_TABLE = None

for candidate in INVENTORY_TABLE_CANDIDATES:

    if spark.catalog.tableExists(candidate):
        INVENTORY_TABLE = candidate
        break

if INVENTORY_TABLE is None:

    raise Exception(
        "Inventory Silver table not found. Checked: "
        + ", ".join(INVENTORY_TABLE_CANDIDATES)
    )

print("Using inventory table:", INVENTORY_TABLE)

# Load datasets.

customers_df = spark.table(TABLES["customers"])
products_df = spark.table(TABLES["products"])
orders_df = spark.table(TABLES["orders"])
order_items_df = spark.table(TABLES["order_items"])
payments_df = spark.table(TABLES["payments"])
shipments_df = spark.table(TABLES["shipments"])
inventory_df = spark.table(INVENTORY_TABLE)

print("\nAll Silver datasets loaded successfully.")


## 4. Display Silver dataset record counts

DATASETS = [
    ("Customers", customers_df),
    ("Products", products_df),
    ("Orders", orders_df),
    ("Order Items", order_items_df),
    ("Payments", payments_df),
    ("Shipments", shipments_df),
    ("Inventory Snapshots", inventory_df)
]

dataset_counts = {}

for name, df in DATASETS:

    count = df.count()

    dataset_counts[name] = count

    print(f"{name}: {count} records")


## 5. Reusable validation report function

results = []

VALID_SEVERITIES = [
    "ERROR",
    "WARNING",
    "INFO"
]

def add_result(
    rule_name,
    dataset_name,
    total_records,
    failed_records,
    description,
    severity="ERROR"
):

    total_records = int(total_records)
    failed_records = int(failed_records)

    severity = severity.upper()

    if severity not in VALID_SEVERITIES:
        raise ValueError(
            f"Invalid severity: {severity}"
        )

    if total_records < 0 or failed_records < 0:
        raise ValueError(
            "Record counts cannot be negative."
        )

    if failed_records > total_records:
        raise ValueError(
            f"Failed records cannot exceed total records: {rule_name}"
        )

    failure_percentage = (
        round(
            failed_records / total_records * 100,
            2
        )
        if total_records > 0
        else 0.0
    )

    if total_records == 0:

        status = "NO_DATA"

    elif severity == "INFO":

        status = "INFO"

    elif failed_records == 0:

        status = "PASS"

    elif severity == "WARNING":

        status = "WARN"

    else:

        status = "FAIL"

    results.append({

        "rule_name": rule_name,

        "dataset_name": dataset_name,

        "total_records": total_records,

        "failed_records": failed_records,

        "failure_percentage": float(
            failure_percentage
        ),

        "status": status,

        "severity": severity,

        "description": description,

        "checked_at": datetime.now()

    })


print("Validation report function ready.")


## 6. Validate order totals against order items

item_totals_df = (

    order_items_df

    .groupBy("order_id")

    .agg(
        F.sum("line_total").alias(
            "calculated_order_total"
        )
    )

)

order_reconciliation_df = (

    orders_df

    .join(
        item_totals_df,
        on="order_id",
        how="inner"
    )

)

invalid_order_total_df = (

    order_reconciliation_df

    .filter(

        F.col("order_total").isNull()

        |

        F.col("calculated_order_total").isNull()

        |

        (
            F.abs(
                F.col("order_total")
                - F.col("calculated_order_total")
            ) > F.lit(TOLERANCE)
        )

    )

)

reconciliation_total = order_reconciliation_df.count()

reconciliation_failed = invalid_order_total_df.count()

add_result(
    "order_total_reconciliation",
    "silver_orders",
    reconciliation_total,
    reconciliation_failed,
    "Order total must match the sum of its valid Silver order item line totals within the configured tolerance."
)

display(
    invalid_order_total_df.select(
        "order_id",
        "order_total",
        "calculated_order_total"
    )
)

print("Orders included in reconciliation:", reconciliation_total)

print(
    "Orders without valid Silver items:",
    orders_df.count() - reconciliation_total
)

## 7. Identify non-cancelled orders without order items

orders_requiring_items_df = (

    orders_df

    .filter(

        F.coalesce(
            F.upper(F.trim(F.col("order_status"))),
            F.lit("UNKNOWN")
        ) != "CANCELLED"

    )

)

valid_item_order_ids_df = (

    order_items_df

    .select("order_id")

    .distinct()

)

orders_without_items_df = (

    orders_requiring_items_df

    .join(
        valid_item_order_ids_df,
        on="order_id",
        how="left_anti"
    )

)

add_result(
    "orders_without_items",
    "silver_orders",
    orders_requiring_items_df.count(),
    orders_without_items_df.count(),
    "Every non-cancelled order must contain at least one valid Silver order item."
)

display(
    orders_without_items_df.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_total"
    )
)


## 8. Validate successful payment amounts against order totals

successful_payments_df = (

    payments_df

    .filter(

        F.upper(
            F.trim(F.col("payment_status"))
        ) == "SUCCESS"

    )

)

payment_totals_df = (

    successful_payments_df

    .groupBy("order_id")

    .agg(

        F.sum("amount").alias(
            "successful_payment_total"
        ),

        F.count("*").alias(
            "successful_payment_count"
        )

    )

)

payment_reconciliation_df = (

    orders_df

    .join(
        payment_totals_df,
        on="order_id",
        how="inner"
    )

)

excess_payment_df = (

    payment_reconciliation_df

    .filter(

        F.col("order_total").isNull()

        |

        F.col("successful_payment_total").isNull()

        |

        (
            F.col("successful_payment_total")
            > F.col("order_total") + F.lit(TOLERANCE)
        )

    )

)

add_result(
    "successful_payment_exceeds_order_total",
    "silver_payments",
    payment_reconciliation_df.count(),
    excess_payment_df.count(),
    "Aggregated successful payments must not exceed the associated order total by more than the configured tolerance."
)

display(
    excess_payment_df.select(
        "order_id",
        "order_total",
        "successful_payment_total"
    )
)


## 9. Identify orders without successful payments

orders_without_successful_payment_df = (

    orders_requiring_items_df

    .join(

        successful_payments_df
        .select("order_id")
        .distinct(),

        on="order_id",

        how="left_anti"

    )

)

add_result(
    "orders_without_successful_payment",
    "silver_orders",
    orders_requiring_items_df.count(),
    orders_without_successful_payment_df.count(),
    "Identifies non-cancelled orders without a successful payment. Pending or unpaid orders may require business investigation.",
    severity="WARNING"
)

display(orders_without_successful_payment_df)


## 10. Identify orders with multiple successful payments

multiple_payments_df = (

    successful_payments_df

    .groupBy("order_id")

    .agg(

        F.count("*").alias(
            "successful_payment_count"
        ),

        F.sum("amount").alias(
            "successful_payment_total"
        )

    )

    .filter(
        F.col("successful_payment_count") > 1
    )

)

successful_payment_order_count = (
    successful_payments_df
    .select("order_id")
    .distinct()
    .count()
)

multiple_payment_order_count = (
    multiple_payments_df.count()
)

add_result(
    "multiple_successful_payments",
    "silver_payments",
    successful_payment_order_count,
    multiple_payment_order_count,
    "Informational check identifying orders with multiple successful payment records.",
    severity="INFO"
)

display(multiple_payments_df)


## 11. Validate payment references against orders

orphan_payments_df = (

    payments_df

    .join(

        orders_df
        .select("order_id")
        .distinct(),

        on="order_id",

        how="left_anti"

    )

)

add_result(
    "payment_without_order",
    "silver_payments",
    payments_df.count(),
    orphan_payments_df.count(),
    "Every payment must reference an existing Silver order."
)

display(orphan_payments_df)


## 12. Validate shipment records against orders

orphan_shipments_df = (

    shipments_df

    .join(

        orders_df
        .select("order_id")
        .distinct(),

        on="order_id",

        how="left_anti"

    )

)

add_result(
    "shipment_without_order",
    "silver_shipments",
    shipments_df.count(),
    orphan_shipments_df.count(),
    "Every shipment must reference an existing Silver order."
)

display(orphan_shipments_df)


 ## 13. Validate delivered shipment dates

invalid_delivery_df = (

    shipments_df

    .filter(

        (

            (
                F.upper(
                    F.trim(F.col("shipment_status"))
                ) == "DELIVERED"
            )

            & F.col("delivered_date").isNull()

        )

        |

        (

            F.col("delivered_date").isNotNull()

            & F.col("shipped_date").isNotNull()

            & (
                F.col("delivered_date")
                < F.col("shipped_date")
            )

        )

    )

)

add_result(
    "shipment_date_consistency",
    "silver_shipments",
    shipments_df.count(),
    invalid_delivery_df.count(),
    "Delivered shipments must have a delivery date, and delivery cannot precede shipment."
)

display(invalid_delivery_df)


## 14. Validate product selling price against product cost

loss_making_products_df = (

    products_df

    .filter(

        F.col("unit_price").isNotNull()

        & F.col("unit_cost").isNotNull()

        & (
            F.col("unit_price") < F.col("unit_cost")
        )

    )

)

add_result(
    "selling_price_below_product_cost",
    "silver_products",
    products_df.count(),
    loss_making_products_df.count(),
    "Identifies products whose selling price is below recorded unit cost.",
    severity="WARNING"
)

display(loss_making_products_df)


## 15. Validate inventory snapshot quantities

invalid_inventory_df = (

    inventory_df

    .filter(

        F.col("quantity_on_hand").isNull()

        |

        (
            F.col("quantity_on_hand") < 0
        )

    )

)

add_result(
    "invalid_inventory_quantity",
    "silver_inventory_snapshots",
    inventory_df.count(),
    invalid_inventory_df.count(),
    "Inventory quantity must be present and cannot be negative."
)

display(invalid_inventory_df)


## 16. Detect duplicate inventory business keys

inventory_key_counts_df = (

    inventory_df

    .groupBy(
        "product_id",
        "warehouse_id",
        "snapshot_date"
    )

    .count()

)

duplicate_inventory_df = (

    inventory_key_counts_df

    .filter(
        F.col("count") > 1
    )

)

duplicate_inventory_rows_df = (

    duplicate_inventory_df

    .select(

        F.sum(
            F.col("count") - F.lit(1)
        ).alias("duplicate_rows")

    )

)

duplicate_inventory_rows = (
    duplicate_inventory_rows_df
    .first()["duplicate_rows"]
)

duplicate_inventory_rows = (
    int(duplicate_inventory_rows)
    if duplicate_inventory_rows is not None
    else 0
)

add_result(
    "duplicate_inventory_business_key",
    "silver_inventory_snapshots",
    inventory_df.count(),
    duplicate_inventory_rows,
    "Product, warehouse, and snapshot date combinations must be unique."
)

display(duplicate_inventory_df)


## 17. Validate order customer references

orphan_orders_df = (

    orders_df

    .join(

        customers_df
        .select("customer_id")
        .distinct(),

        on="customer_id",

        how="left_anti"

    )

)

add_result(
    "order_without_customer",
    "silver_orders",
    orders_df.count(),
    orphan_orders_df.count(),
    "Every order must reference an existing Silver customer."
)

display(orphan_orders_df)

## 18. Validate order item product references

orphan_order_items_product_df = (

    order_items_df

    .join(

        products_df
        .select("product_id")
        .distinct(),

        on="product_id",

        how="left_anti"

    )

)

add_result(
    "order_item_without_product",
    "silver_order_items",
    order_items_df.count(),
    orphan_order_items_product_df.count(),
    "Every order item must reference an existing Silver product."
)

display(orphan_order_items_product_df)


## 19. Validate order item order references

orphan_order_items_order_df = (

    order_items_df

    .join(

        orders_df
        .select("order_id")
        .distinct(),

        on="order_id",

        how="left_anti"

    )

)

add_result(
    "order_item_without_order",
    "silver_order_items",
    order_items_df.count(),
    orphan_order_items_order_df.count(),
    "Every order item must reference an existing Silver order."
)

display(orphan_order_items_order_df)


## 20. Validate inventory product references

orphan_inventory_df = (

    inventory_df

    .join(

        products_df
        .select("product_id")
        .distinct(),

        on="product_id",

        how="left_anti"

    )

)

add_result(
    "inventory_without_product",
    "silver_inventory_snapshots",
    inventory_df.count(),
    orphan_inventory_df.count(),
    "Every inventory snapshot must reference an existing Silver product."
)

display(orphan_inventory_df)


## 21. Create business rule validation report

report_df = spark.createDataFrame(results)

display(
    report_df.orderBy(
        "dataset_name",
        "rule_name"
    )
)

print("Business rule validation report generated.")


## 22. Save report as Delta table

(

    report_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(REPORT_TABLE)

)

print(
    f"Business rule report saved: {REPORT_TABLE}"
)


## 23. Generate dataset-level validation summary

summary_df = (

    report_df

    .groupBy("dataset_name")

    .agg(

        F.count("*").alias(
            "total_rules"
        ),

        F.sum(
            F.when(
                F.col("status") == "PASS", 1
            ).otherwise(0)
        ).alias(
            "passed_rules"
        ),

        F.sum(
            F.when(
                F.col("status") == "FAIL", 1
            ).otherwise(0)
        ).alias(
            "failed_rules"
        ),

        F.sum(
            F.when(
                F.col("status") == "WARN", 1
            ).otherwise(0)
        ).alias(
            "warning_rules"
        ),

        F.sum(
            F.when(
                F.col("status") == "INFO", 1
            ).otherwise(0)
        ).alias(
            "informational_rules"
        ),

        F.sum(
            F.when(
                F.col("status") == "NO_DATA", 1
            ).otherwise(0)
        ).alias(
            "no_data_rules"
        )

    )

    .withColumn(

        "overall_status",

        F.when(
            F.col("failed_rules") > 0,
            "FAIL"
        )

        .when(
            F.col("no_data_rules") > 0,
            "CHECK"
        )

        .when(
            F.col("warning_rules") > 0,
            "WARN"
        )

        .otherwise("PASS")

    )

    .orderBy("dataset_name")

)

display(summary_df)


# 24. Final validation summary

report_df = spark.table(REPORT_TABLE)

total_rules = report_df.count()

passed_rules = (

    report_df

    .filter(
        F.col("status") == "PASS"
    )

    .count()

)

failed_rules = (

    report_df

    .filter(
        F.col("status") == "FAIL"
    )

    .count()

)

warning_rules = (

    report_df

    .filter(
        F.col("status") == "WARN"
    )

    .count()

)

informational_rules = (

    report_df

    .filter(
        F.col("status") == "INFO"
    )

    .count()

)

no_data_rules = (

    report_df

    .filter(
        F.col("status") == "NO_DATA"
    )

    .count()

)

print("\n")
print("=" * 70)
print("BUSINESS RULE VALIDATION FINAL SUMMARY")
print("=" * 70)

print(f"Total rules         : {total_rules}")
print(f"Passed rules        : {passed_rules}")
print(f"Failed rules        : {failed_rules}")
print(f"Warning rules       : {warning_rules}")
print(f"Informational rules : {informational_rules}")
print(f"No-data rules       : {no_data_rules}")

print("-" * 70)

if failed_rules > 0:

    print("RESULT: BUSINESS RULE FAILURES FOUND.")

elif no_data_rules > 0:

    print("RESULT: CHECK RULES WITH NO DATA.")

elif warning_rules > 0:

    print("RESULT: VALIDATION PASSED WITH WARNINGS.")

else:

    print("RESULT: ALL BUSINESS RULES PASSED.")

print("-" * 70)

print("Inventory table:", INVENTORY_TABLE)
print("Report table:", REPORT_TABLE)

print("=" * 70)

# COMMAND ----------



| Rule | What it checks |
|---|---|
| order_total_reconciliation | Verifies order total against the sum of order-item totals. |
| orders_without_items | Identifies orders that have no associated order items. |
| successful_payment_exceeds_order_total | Identifies successful payment amounts exceeding the order total. |
| multiple_successful_payment` | Informational check for multiple successful payments against the same order. |
| shipment_without_order | Identifies shipments referencing non-existent orders. |
| shipment_date_consistency | Validates shipment and delivery date logic. |
| selling_price_below_product_cost | Identifies products sold below their purchase cost, indicating potential losses. |
| invalid_inventory_quantity | Identifies negative or missing inventory quantities. |
| duplicate_inventory_business_key | Identifies duplicate inventory snapshots using the composite business key. |
| order_without_customer | Identifies orders referencing non-existent customers. |
| order_item_without_product | Identifies order items referencing non-existent products. |
| order_item_without_order  | Identifies order items referencing non-existent orders. |

In [0]:
# Display all failed business rules

failed_rules_df = (
    spark.table(
        "workspace.ecommerce_silver.business_rule_validation_report"
    )
    .filter(F.col("status") == "FAIL")
    .orderBy("rule_name")
)

display(failed_rules_df)

In [0]:
# Display the number of records failing each rule

failed_rules_df.select(
    "rule_name",
    "dataset_name",
    "failed_records",
    "failure_percentage",
    "description"
).show(50, truncate=False)

In [0]:
# ============================================================
# BUSINESS RULE FAILURE DIAGNOSTICS
# ============================================================

# 1. Order items referencing missing orders
orphan_order_items = (
    order_items_df
    .join(
        orders_df.select("order_id").distinct(),
        "order_id",
        "left_anti"
    )
)

print("1. ORDER ITEMS WITHOUT ORDERS")
display(
    orphan_order_items.select(
        "order_item_id", "order_id", "product_id"
    )
)

# 2. Order items referencing missing products
orphan_products = (
    order_items_df
    .join(
        products_df.select("product_id").distinct(),
        "product_id",
        "left_anti"
    )
)

print("2. ORDER ITEMS WITHOUT PRODUCTS")
display(
    orphan_products.select(
        "order_item_id", "order_id", "product_id"
    )
)

# 3. Orders referencing missing customers
orphan_customers = (
    orders_df
    .join(
        customers_df.select("customer_id").distinct(),
        "customer_id",
        "left_anti"
    )
)

print("3. ORDERS WITHOUT CUSTOMERS")
display(
    orphan_customers.select(
        "order_id", "customer_id", "order_status"
    )
)

# 4. Check whether missing customers exist in quarantine
quarantine_df = spark.table(
    "workspace.ecommerce_silver.quarantine_customers"
)

orders_with_quarantined_customers = (
    orphan_customers
    .join(
        quarantine_df.select("customer_id").distinct(),
        "customer_id",
        "inner"
    )
)

print("4. ORDERS LINKED TO QUARANTINED CUSTOMERS")
display(
    orders_with_quarantined_customers.select(
        "order_id", "customer_id"
    )
)

# 5. Orders without order items
orders_without_items = (
    orders_df
    .join(
        order_items_df.select("order_id").distinct(),
        "order_id",
        "left_anti"
    )
)

print("5. ORDERS WITHOUT ORDER ITEMS")
display(
    orders_without_items.select(
        "order_id", "customer_id", "order_total"
    )
)

# 6. Order total mismatches
item_totals = (
    order_items_df
    .groupBy("order_id")
    .agg(
        F.sum("line_total").alias("calculated_order_total")
    )
)

order_mismatches = (
    orders_df
    .join(item_totals, "order_id", "inner")
    .filter(
        F.abs(
            F.col("order_total") -
            F.col("calculated_order_total")
        ) > F.lit(0.02)
    )
)

print("6. ORDER TOTAL MISMATCHES")
display(
    order_mismatches.select(
        "order_id",
        "order_total",
        "calculated_order_total"
    )
)

# 7. Shipments referencing missing orders
orphan_shipments = (
    shipments_df
    .join(
        orders_df.select("order_id").distinct(),
        "order_id",
        "left_anti"
    )
)

print("7. SHIPMENTS WITHOUT ORDERS")
display(
    orphan_shipments.select(
        "shipment_id", "order_id", "shipment_status"
    )
)

print("DIAGNOSTIC CHECKS COMPLETED")

In [0]:
%sql
SELECT
    rule_name,
    dataset_name,
    total_records,
    failed_records,
    failure_percentage,
    status,
    severity,
    description
FROM workspace.ecommerce_silver.business_rule_validation_report
WHERE status IN ('FAIL', 'WARN')
ORDER BY
    CASE status
        WHEN 'FAIL' THEN 1
        WHEN 'WARN' THEN 2
    END,
    dataset_name,
    rule_name;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.business_rule_validation_report
ORDER BY dataset_name, rule_name;


**A. Find order items referencing missing orders**

In [0]:
%sql

SELECT oi.*
FROM workspace.ecommerce_silver.silver_order_items oi
LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
    ON oi.order_id = o.order_id;


**B. Find order items referencing missing products**

In [0]:
%sql

SELECT oi.*
FROM workspace.ecommerce_silver.silver_order_items oi
LEFT ANTI JOIN workspace.ecommerce_silver.silver_products p
    ON oi.product_id = p.product_id;


**C. Find inventory snapshots referencing missing products**

In [0]:
%sql

SELECT i.*
FROM workspace.ecommerce_silver.silver_inventory i
LEFT ANTI JOIN workspace.ecommerce_silver.silver_products p
    ON i.product_id = p.product_id;


**D. Find orders referencing missing customers**

In [0]:
%sql

SELECT o.*
FROM workspace.ecommerce_silver.silver_orders o
LEFT ANTI JOIN workspace.ecommerce_silver.silver_customers c
    ON o.customer_id = c.customer_id;


**E. Find the order without valid items**

In [0]:
%sql

SELECT
    o.order_id,
    o.order_status,
    o.order_total
FROM workspace.ecommerce_silver.silver_orders o
LEFT ANTI JOIN workspace.ecommerce_silver.silver_order_items oi
    ON o.order_id = oi.order_id
WHERE LOWER(TRIM(o.order_status)) <> 'cancelled';


**F. Find payments and shipments referencing missing orders**

In [0]:
%sql
SELECT p.*
FROM workspace.ecommerce_silver.silver_payments p
LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
    ON p.order_id = o.order_id;


**Investigate the missing products first**

In [0]:
%sql
SELECT
    b.product_id,
    COUNT(*) AS bronze_record_count
FROM workspace.ecommerce_bronze.bronze_products b
LEFT ANTI JOIN workspace.ecommerce_silver.silver_products s
    ON b.product_id = s.product_id
GROUP BY b.product_id
ORDER BY b.product_id;


**Identify the single order without items**

In [0]:
%sql
SELECT
    o.order_id,
    o.order_status,
    o.order_total
FROM workspace.ecommerce_silver.silver_orders o
LEFT ANTI JOIN workspace.ecommerce_silver.silver_order_items oi
    ON o.order_id = oi.order_id
WHERE UPPER(TRIM(o.order_status)) <> 'CANCELLED';


**Two more outputs are still needed**

In [0]:
%sql
SELECT p.*
FROM workspace.ecommerce_silver.silver_payments p
LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
    ON p.order_id = o.order_id;


**Run this SQL to investigate missing order references**

In [0]:
%sql
SELECT
    p.payment_id,
    p.order_id,
    p.payment_method,
    p.payment_status,
    p.amount
FROM workspace.ecommerce_silver.silver_payments p
LEFT ANTI JOIN workspace.ecommerce_silver.silver_orders o
    ON p.order_id = o.order_id
ORDER BY p.order_id;


**Check the business-rule report again**

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.business_rule_validation_report;

In [0]:
# --------------------------------------------------
# CUSTOMER SILVER RECONCILIATION
# --------------------------------------------------

print("1. Missing customer IDs")
display(missing_silver_customers)

print("2. Bronze matching records")
display(
    bronze_customers.join(
        missing_silver_customers,
        on="customer_id",
        how="inner"
    )
)

print("3. Silver customer table count")
print(silver_customers.count())

print("4. Bronze customer table count")
print(bronze_customers.count())

print("5. Customer quarantine count")
print(customer_quarantine.count())

print("6. Customer quarantine reasons")
display(
    customer_quarantine
    .groupBy("quarantine_reason")
    .count()
)

In [0]:
bronze_customer_ids = bronze_customers.select(
    "customer_id"
).distinct()

silver_customer_ids = silver_customers.select(
    "customer_id"
).distinct()

missing_from_silver = bronze_customer_ids.join(
    silver_customer_ids,
    on="customer_id",
    how="left_anti"
)

print("Bronze customer IDs missing from Silver:")
print(missing_from_silver.count())

display(missing_from_silver)

In [0]:
# --------------------------------------------------
# CUSTOMER COUNT RECONCILIATION
# --------------------------------------------------

print("BRONZE TOTAL ROWS:", bronze_customers.count())

print(
    "BRONZE DISTINCT CUSTOMER IDS:",
    bronze_customers.select("customer_id").distinct().count()
)

print("SILVER TOTAL ROWS:", silver_customers.count())

print(
    "SILVER DISTINCT CUSTOMER IDS:",
    silver_customers.select("customer_id").distinct().count()
)

print("BRONZE DUPLICATE CUSTOMER IDS:")

bronze_duplicates = (
    bronze_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

display(
    bronze_duplicates.orderBy(F.col("count").desc())
)

print("ALL BRONZE CUSTOMERS MISSING FROM SILVER:")

display(missing_from_silver)

In [0]:
print("COMPLETE MISSING CUSTOMER RECORDS:")

display(
    bronze_customers.join(
        missing_from_silver,
        on="customer_id",
        how="inner"
    )
)